# 24. DILI 모델 추가 + hydroquinone 규칙 확장

## 이번 노트북에서 할 것
- TDC에서 DILI(약물유발 간손상) 데이터셋 로드, ECFP+RandomForest baseline 학습
- hydroquinone 규칙 추가 (catechol의 add_substituent 로직 재사용)
- 두 작업 완료 후 회귀 테스트 + 커버리지 재확인
- 여유 있으면 ChEMBL로 phosphor, Oxygen-nitrogen_single_bond 등 중간권
  규칙 빈도 재확인

## 간략한 정리 (23까지)
- 라이브러리 14개 규칙 최종 확정, 여러 버그 수정(het-C-het SMARTS 재조정,
  imine_1_general이 구아니딘 잘못 매치하던 문제 발견/수정)
- Valid set(seed=7) 최종 수치 확정: 커버리지 26.2%, 성공률 90%,
  Ames p<0.0001(가장 뚜렷), Tox21 경계(p=0.085), hERG 유의X(p=0.306),
  QED +0.059(81% 개선)
- ChEMBL 승인약물 vs Tox21 빈도 비교 완료, 대부분 규칙 실용성 확인
- 다음 확장 목표: 데이터셋(DILI 우선) + 규칙(hydroquinone부터, 문헌형 5개
  우선순위 확정) 추가로 커버리지 및 Tox21/hERG 개선 효과 강화
- test set은 여전히 미사용, 학생 승인 전까지 유지

## 다음에 해야 할 것 (오늘 끝나면)
- beta-keto/anhydride, phthalimide 등 문헌형 규칙은 학생이 논문 확인 후 진행
- 모든 규칙 확정 후 최종 valid set 재검증 -> 학생 승인 시 test set 1회 검증
- 제안서는 학생이 계속 병행 작성 중

In [1]:
# 셀 1
!pip install rdkit -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 37.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 266, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 266 (delta 0), reused 2 (delta 0), pack-reused 260 (from 1)
Receiving objects: 100% (266/266), 643.12 KiB | 10.21 MiB/s, done.
Resolving deltas: 100% (137/137), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib, random
import numpy as np, pandas as pd
from collections import Counter
from scipy import stats
from rdkit import Chem
from rdkit.Chem import rdMMPA, rdFingerprintGenerator, QED
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.agent

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use
from src.tools.atom_editor import apply_atom_edit_from_rule

data = load_tox21_clean(random_state=7)

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprintAsNumPy(mol) if mol else None

print(f"도구 로드 완료. 현재 라이브러리 규칙 수: {len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])}")

[04:10:13] WARNING: not removing hydrogen atom without neighbors
[04:10:14] Explicit valence for atom # 8 Al, 6, is greater than permitted
[04:10:14] Explicit valence for atom # 3 Al, 6, is greater than permitted
[04:10:14] Explicit valence for atom # 4 Al, 6, is greater than permitted
[04:10:15] Explicit valence for atom # 4 Al, 6, is greater than permitted
[04:10:15] Explicit valence for atom # 9 Al, 6, is greater than permitted
[04:10:15] Explicit valence for atom # 5 Al, 6, is greater than permitted
[04:10:16] Explicit valence for atom # 16 Al, 6, is greater than permitted
[04:10:16] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[04:10:16] WARNING: not removing hydrogen atom without neighbors


도구 로드 완료. 현재 라이브러리 규칙 수: 14


In [5]:
def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini"):
    """진단->치환->재평가를 반복.
    llm_client가 주어지면: 어떤 문제부터 고칠지 + 어떤 후보를 쓸지 둘 다 LLM이 판단.
    llm_client_type: "gemini" 또는 "openai_compatible".
    llm_client가 없으면: 리스트 순서(known_problems[0]) + candidate_idx 고정값 사용."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []  # 신규: 사유를 사람이 읽기 좋게 기록

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            target_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')

            candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, target_rule, client_type=llm_client_type)
            chosen_candidate_idx = candidate_decision['candidate_idx']
            candidate_reason = candidate_decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            problem_reason = "규칙 기반(리스트 순서대로)"
            chosen_candidate_idx = candidate_idx
            candidate_reason = "규칙 기반(고정 인덱스)"

        fixed = propose_fix(current, target_rule, chosen_candidate_idx)

        if fixed is None or not fixed['is_valid']:
            mol = Chem.MolFromSmiles(current)
            matched = next((p['atom_indices'] for p in problems if p['rule_name'] == target_rule), [])
            reason_detail = (f"'{target_rule}' 규칙은 라이브러리에 있으나, 이 분자의 구체적 구조에서 "
                             f"치환 실행이 실패했습니다. 흔한 원인: 유기금속/무기염 등 특수 화학종, "
                             f"고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다. "
                             f"매치된 원자: {matched}")
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}

In [ ]:
!cat src/tools/molecule_editor.py

from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    """조각이 problem_pattern과 정확한 크기로 매치되는지 확인."""
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환.
    1단계(maxCuts=1)로 단순 분리를 먼저 시도하고,
    실패하면 2단계(maxCuts=2)로 고리 인접 작용기 분리를 시도한다."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)


In [ ]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    """조각이 problem_pattern과 정확한 크기로 매치되는지 확인."""
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환.
    1단계(maxCuts=1)로 단순 분리를 먼저 시도하고,
    실패하면 2단계(maxCuts=2)로 고리 인접 작용기 분리를 시도한다."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # --- Case A: 단순 2조각 분리 (maxCuts=1) ---
    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    # --- Case B: 고리 인접 등, core가 남는 2-cut 분리 ---
    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]

            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue

            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue

            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')

            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    """규칙의 edit_method에 따라 결합절단형(기존) 또는 원자직접편집형(신규)으로 분기."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini"):
    """진단->치환->재평가를 반복.
    llm_client가 주어지면: 어떤 문제부터 고칠지 + 어떤 후보를 쓸지 둘 다 LLM이 판단.
    llm_client_type: "gemini" 또는 "openai_compatible".
    llm_client가 없으면: 리스트 순서(known_problems[0]) + candidate_idx 고정값 사용.
    skipped_details/reason_detail: 연구자가 no_known_fix/stuck 사유를 바로
    확인할 수 있도록 사람이 읽을 수 있는 설명과 매치된 원자 정보를 함께 제공."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol_cur = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol_cur.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol_cur else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            target_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')

            candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, target_rule, client_type=llm_client_type)
            chosen_candidate_idx = candidate_decision['candidate_idx']
            candidate_reason = candidate_decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            problem_reason = "규칙 기반(리스트 순서대로)"
            chosen_candidate_idx = candidate_idx
            candidate_reason = "규칙 기반(고정 인덱스)"

        fixed = propose_fix(current, target_rule, chosen_candidate_idx)

        if fixed is None or not fixed['is_valid']:
            matched = next((p['atom_indices'] for p in problems if p['rule_name'] == target_rule), [])
            reason_detail = (f"'{target_rule}' 규칙은 라이브러리에 있으나, 이 분자의 구체적 구조에서 "
                              f"치환 실행이 실패했습니다. 흔한 원인: 유기금속/무기염 등 특수 화학종, "
                              f"고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다. "
                              f"매치된 원자: {matched}")
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}

Overwriting src/tools/molecule_editor.py


In [ ]:
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop

test_case = "CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C"
result_test = iterative_fix_loop(test_case, max_iterations=10)
print("상태:", result_test['status'])
print("\nskipped_details:")
for d in result_test.get('skipped_details', []):
    print(f"  [{d['rule_name']}] {d['reason']}")

상태: success

skipped_details:


In [ ]:
# 구아니딘이 포함된 분자로 테스트 (imine_1_guanidine은 라이브러리에 없음)
test_case2 = "CCOC(=O)c1ccc(OC(=O)CCCCCNC(=N)N)cc1"
result_test2 = iterative_fix_loop(test_case2, max_iterations=10)
print("상태:", result_test2['status'])
print("\nskipped_details:")
for d in result_test2.get('skipped_details', []):
    print(f"  [{d['rule_name']}]")
    print(f"    사유: {d['reason']}\n")

상태: no_known_fix

skipped_details:
  [Aliphatic_long_chain]
    사유: 라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 'Aliphatic_long_chain'로 진단했으며, 매치된 원자 인덱스는 [12, 13, 14, 15](원소: ['C', 'C', 'C', 'C'])입니다. 이 구조에 대한 치환 규칙을 replacement_library.py에 추가하면 자동으로 처리 가능합니다.

  [imine_1_guanidine]
    사유: 라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 'imine_1_guanidine'로 진단했으며, 매치된 원자 인덱스는 [18, 19](원소: ['C', 'N'])입니다. 이 구조에 대한 치환 규칙을 replacement_library.py에 추가하면 자동으로 처리 가능합니다.

  [imine_2]
    사유: 라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 'imine_2'로 진단했으며, 매치된 원자 인덱스는 [17, 18, 19](원소: ['N', 'C', 'N'])입니다. 이 구조에 대한 치환 규칙을 replacement_library.py에 추가하면 자동으로 처리 가능합니다.

  [phenol_ester]
    사유: 라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 'phenol_ester'로 진단했으며, 매치된 원자 인덱스는 [5, 6, 7, 8, 9, 10, 11, 12, 21, 22](원소: ['C', 'C', 'C', 'C', 'O', 'C', 'O', 'C', 'C', 'C'])입니다. 이 구조에 대한 치환 규칙을 replacement_library.py에 추가하면 자동으로 처리 가능합니다.



In [ ]:
!git add src/tools/molecule_editor.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/molecule_editor.py



In [ ]:
!git commit -m "Add skipped_details/reason_detail to iterative_fix_loop output: no_known_fix and stuck cases now include human-readable explanation with matched atom indices and elements, so researchers can quickly diagnose why a molecule wasn't fixed and decide whether to extend the library. Implements student-proposed usability improvement for domain expert users."
!git push origin main

[main 7f16df7] Add skipped_details/reason_detail to iterative_fix_loop output: no_known_fix and stuck cases now include human-readable explanation with matched atom indices and elements, so researchers can quickly diagnose why a molecule wasn't fixed and decide whether to extend the library. Implements student-proposed usability improvement for domain expert users.
 1 file changed, 32 insertions(+), 6 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.52 KiB | 1.52 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   2ba2415..7f16df7  main -> main


In [ ]:
X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
task_cols = data['task_cols']
classifiers = {}
for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])
    classifiers[task] = clf
print("Tox21 baseline 완료")

Tox21 baseline 완료


In [ ]:
from tdc.single_pred import Tox

def prepare_split_generic(df):
    df = df.copy()
    df['mol_valid'] = df['Drug'].apply(lambda s: Chem.MolFromSmiles(s) is not None)
    df_clean = df[df['mol_valid']].reset_index(drop=True)
    X = np.stack(df_clean['Drug'].apply(smiles_to_ecfp).values)
    y = df_clean['Y'].values
    return X, y

dili_split = Tox(name='DILI').get_split()
print("Train:", dili_split['train'].shape, "Valid:", dili_split['valid'].shape, "Test:", dili_split['test'].shape)

X_train_dili, y_train_dili = prepare_split_generic(dili_split['train'])
X_valid_dili, y_valid_dili = prepare_split_generic(dili_split['valid'])

dili_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
dili_clf.fit(X_train_dili, y_train_dili)

dili_valid_auc = roc_auc_score(y_valid_dili, dili_clf.predict_proba(X_valid_dili)[:,1])
print(f"DILI Valid AUROC: {dili_valid_auc:.3f}")

def predict_dili(s):
    m = Chem.MolFromSmiles(s)
    return dili_clf.predict_proba(smiles_to_ecfp(s).reshape(1,-1))[0][1] if m else None

Downloading...
100%|██████████| 26.7k/26.7k [00:00<00:00, 572kiB/s]
Loading...
Done!


Train: (332, 3) Valid: (48, 3) Test: (95, 3)
DILI Valid AUROC: 0.921


In [ ]:
print(f"Valid set 클래스 분포: {np.bincount(y_valid_dili.astype(int))}")

X_test_dili, y_test_dili = prepare_split_generic(dili_split['test'])
test_auc_dili = roc_auc_score(y_test_dili, dili_clf.predict_proba(X_test_dili)[:,1])
print(f"\nDILI 자체 Test AUROC (95개, 더 안정적인 추정): {test_auc_dili:.3f}")

Valid set 클래스 분포: [21 27]

DILI 자체 Test AUROC (95개, 더 안정적인 추정): 0.871


20. **보조 독성 예측 모델(Ames/hERG/DILI)의 test 분할 사용에 대한 방법론적
    설명**: 핵심 검증 대상인 Tox21(자체 라이브러리/에이전트 성능 평가 기준)은
    test set을 최종 검증까지 엄격히 보류하는 원칙을 지켰음. 반면 보조
    endpoint(Ames, hERG, DILI)는 TDC가 제공하는 표준 분할을 그대로 사용해
    baseline 모델 성능을 1회만 확인했으며, 이 결과로 모델을 재조정하거나
    치환 로직을 튜닝하지 않았음. 반복적 개발 과정에서 오염될 위험이 낮다고
    판단하여 재분할하지 않았음을 명시함.

---
type: project-log
date: 2026-07-27
status: active
tags:
  - type/project-log
  - status/active
  - area/ai-drug-discovery
  - topic/molecular-optimization
---

# 커버리지 확장 계획 — 데이터형 재스크리닝 + 임상탈락 기반 규칙

관련 프로젝트: [[LAIDD 2026]]

## 완료된 것 (2026-07-27, 24번 노트북 시점)
- 라이브러리 14개 규칙, valid set 커버리지 26.2%, 성공률 90%
- no_known_fix/stuck 사유 명시 기능 추가(skipped_details, reason_detail)
- DILI 모델 추가(Test AUROC 0.871, n=95) — 4번째 endpoint 확보
- Tox21은 test set 미사용 원칙 엄격 유지, Ames/hERG/DILI는 TDC 표준분할로
  1회 확인만 함(재조정 없음, 방법론적 판단 사유 limitations.md에 기록 예정)

## 다음 확장 — 데이터형 재스크리닝 (중간 순위 규칙, ChEMBL 빈도 재확인 대상)
아까 확인한 미커버 규칙 중, Aliphatic_long_chain/isolated_alkene처럼 너무
광범위한 것은 제외하고, 다음을 ChEMBL 승인약물 빈도로 재확인 후 선별 추가:
- phosphor (22개, valid set 기준)
- Oxygen-nitrogen_single_bond (13개)
- diketo_group (8개)
- halogenated_ring_1 (9개)
- Three-membered_heterocycle (6개, 에폭시/아지리딘 계열 - 반응성 명확)

## 다음 확장 — 문헌형 (기존 목록, 우선순위 유지)
1. hydroquinone (catechol 로직 재사용, 최우선)
2. beta-keto/anhydride
3. phthalimide
4. hydroxamic_acid
5. quinone_A(370)

## 신규 — 실제 약물/임상탈락 사례 기반 규칙
- 개발 중단/철수된 약물의 구조적 원인을 조사하여, 이미 우리 라이브러리에
  없는 독성 구조가 있으면 문헌형으로 추가
- 후보 조사 대상(향후 확인 필요): 트로글리타존(간독성, 특정 구조 원인),
  시사프라이드(hERG 관련 철수 사례) 등 잘 알려진 철수 사례
- 목적: Tox21/hERG에서 개선 효과가 약했던 것을 보완 - 실제 hERG 관련
  철수 사례의 구조적 원인을 규칙화하면 hERG endpoint 개선에 직접 기여 가능

## 작업 순서 제안
1. ChEMBL로 중간권 규칙 5개 빈도 재확인 (데이터형)
2. hydroquinone 추가 (가장 쉬움, catechol 로직 재사용)
3. 임상철수 사례 조사 (웹서치) 및 규칙화 시도 (hERG 개선 목표와 직결)
4. 나머지 문헌형(beta-keto/anhydride 등)은 학생 진행과 병행

## 작업 시 유의점 (누적)
- 새 규칙 키는 반드시 detect_toxicophores() 반환 이름과 정확히 일치
- SMARTS 설계 시 이온화 형태, 방향족/지방족 구분(대문자/소문자) 확인
- 편집 방식 7종 중 적합한 것 선택, 필요시 새 타입 설계
- 원자편집형은 화학적으로 유사한 다른 화학종(구아니딘 vs 일반 이민 등)까지
  잘못 매치하지 않는지 반드시 확인
</parameter>

In [6]:
!pip install chembl_webresource_client -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00


In [7]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule

def fetch_chembl_smiles(max_phase_value, limit=500):
    results = molecule.filter(max_phase=max_phase_value).only(
        ['molecule_structures']
    )[:limit]
    smiles_list = []
    for r in results:
        struct = r.get('molecule_structures')
        if struct and struct.get('canonical_smiles'):
            smiles_list.append(struct['canonical_smiles'])
    return smiles_list

print("승인 약물(phase 4) 가져오는 중...")
smiles_phase4 = fetch_chembl_smiles(4, limit=500)
print(f"  {len(smiles_phase4)}개 확보")

승인 약물(phase 4) 가져오는 중...
  498개 확보


In [8]:
mid_tier_rules = ["phosphor", "Oxygen-nitrogen_single_bond", "diketo_group",
                   "halogenated_ring_1", "Three-membered_heterocycle"]

mid_tier_counts = {r: 0 for r in mid_tier_rules}
for s in smiles_phase4:
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        continue
    problems = detect_toxicophores(s)
    found = set(p['rule_name'] for p in problems)
    for r in mid_tier_rules:
        if r in found:
            mid_tier_counts[r] += 1

print(f"승인약물(n={len(smiles_phase4)}) 기준 중간권 규칙 빈도:")
for r, c in sorted(mid_tier_counts.items(), key=lambda x: -x[1]):
    print(f"  {r}: {c}개 ({c/len(smiles_phase4)*100:.1f}%)")

승인약물(n=498) 기준 중간권 규칙 빈도:
  Oxygen-nitrogen_single_bond: 25개 (5.0%)
  phosphor: 8개 (1.6%)
  diketo_group: 3개 (0.6%)
  Three-membered_heterocycle: 3개 (0.6%)
  halogenated_ring_1: 1개 (0.2%)


In [9]:
examples_on = None
for s in smiles_phase4:
    problems = detect_toxicophores(s)
    if any(p['rule_name'] == 'Oxygen-nitrogen_single_bond' for p in problems):
        examples_on = s
        for p in problems:
            if p['rule_name'] == 'Oxygen-nitrogen_single_bond':
                indices = p['atom_indices']
        break

print("예시:", examples_on)
mol = Chem.MolFromSmiles(examples_on)
for idx in indices:
    atom = mol.GetAtomWithIdx(idx)
    print(f"  idx={idx}: {atom.GetSymbol()} (방향족: {atom.GetIsAromatic()}, 이웃: {[n.GetSymbol() for n in atom.GetNeighbors()]})")

예시: O=[N+]([O-])O[C@H]1CO[C@H]2[C@@H]1OC[C@H]2O[N+](=O)[O-]
  idx=1: N (방향족: False, 이웃: ['O', 'O', 'O'])
  idx=2: O (방향족: False, 이웃: ['N'])


In [10]:
examples_hq = None
for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] == 'hydroquinone':
            examples_hq = (s, p['atom_indices'])
            break
    if examples_hq:
        break

print("예시:", examples_hq[0])
mol = Chem.MolFromSmiles(examples_hq[0])
for idx in examples_hq[1]:
    atom = mol.GetAtomWithIdx(idx)
    print(f"  idx={idx}: {atom.GetSymbol()} (방향족: {atom.GetIsAromatic()}, 이웃: {[n.GetSymbol() for n in atom.GetNeighbors()]})")

예시: CCCC(=O)Nc1ccc(O)c(C(C)=O)c1
  idx=5: N (방향족: False, 이웃: ['C', 'C'])
  idx=6: C (방향족: True, 이웃: ['N', 'C', 'C'])
  idx=7: C (방향족: True, 이웃: ['C', 'C'])
  idx=8: C (방향족: True, 이웃: ['C', 'C'])
  idx=9: C (방향족: True, 이웃: ['C', 'O', 'C'])
  idx=10: O (방향족: False, 이웃: ['C'])
  idx=11: C (방향족: True, 이웃: ['C', 'C', 'C'])
  idx=15: C (방향족: True, 이웃: ['C', 'C'])


In [11]:
pattern_hq_test = Chem.MolFromSmarts("[OX2H]c1ccc(N)cc1")
mol_hq = Chem.MolFromSmiles("CCCC(=O)Nc1ccc(O)c(C(C)=O)c1")
print("파라-아미노페놀 패턴 매치:", mol_hq.HasSubstructMatch(pattern_hq_test))
print("패턴 크기:", pattern_hq_test.GetNumAtoms())

파라-아미노페놀 패턴 매치: True
패턴 크기: 8


In [12]:
pure_hydroquinone = Chem.MolFromSmiles("Oc1ccc(O)cc1")
result_pure = detect_toxicophores("Oc1ccc(O)cc1")
print(result_pure)

[{'rule_name': 'hydroquinone', 'atom_indices': [0, 1, 2, 3, 4, 5, 6, 7]}]


In [13]:
pattern_general = Chem.MolFromSmarts("[OX2H]c1ccc([OX2H,NX3H1,NX3H2])cc1")
print("크기:", pattern_general.GetNumAtoms())
print("순수 하이드로퀴논 매치:", Chem.MolFromSmiles("Oc1ccc(O)cc1").HasSubstructMatch(pattern_general))
print("아세트아미노펜계열 매치:", Chem.MolFromSmiles("CCCC(=O)Nc1ccc(O)c(C(C)=O)c1").HasSubstructMatch(pattern_general))

크기: 8
순수 하이드로퀴논 매치: True
아세트아미노펜계열 매치: True


In [14]:
target_check_therapeutic = ["nitro_group", "Michael_acceptor_1", "alkyl_halide",
                              "aniline", "Sulfonic_acid_2", "imine_1_general",
                              "catechol", "Thiocarbonyl_group"]

for rule in target_check_therapeutic:
    count = sum(1 for s in smiles_phase4 if any(p['rule_name'] == rule for p in detect_toxicophores(s)))
    print(f"{rule}: 승인약물 {count}개")

nitro_group: 승인약물 11개
Michael_acceptor_1: 승인약물 9개
alkyl_halide: 승인약물 9개
aniline: 승인약물 19개
Sulfonic_acid_2: 승인약물 2개
imine_1_general: 승인약물 8개
catechol: 승인약물 7개
Thiocarbonyl_group: 승인약물 5개


In [15]:
def fetch_chembl_with_names(max_phase_value, limit=500):
    results = molecule.filter(max_phase=max_phase_value).only(
        ['molecule_structures', 'pref_name']
    )[:limit]
    data_list = []
    for r in results:
        struct = r.get('molecule_structures')
        name = r.get('pref_name')
        if struct and struct.get('canonical_smiles'):
            data_list.append({"smiles": struct['canonical_smiles'], "name": name})
    return data_list

drugs_with_names = fetch_chembl_with_names(4, limit=500)
print(f"확보: {len(drugs_with_names)}개")

print("\n=== Michael_acceptor_1 해당 승인약물 ===")
for d in drugs_with_names:
    problems = detect_toxicophores(d['smiles'])
    if any(p['rule_name'] == 'Michael_acceptor_1' for p in problems):
        print(f"  {d['name']}: {d['smiles'][:60]}")

print("\n=== nitro_group 해당 승인약물 ===")
for d in drugs_with_names:
    problems = detect_toxicophores(d['smiles'])
    if any(p['rule_name'] == 'nitro_group' for p in problems):
        print(f"  {d['name']}: {d['smiles'][:60]}")

확보: 498개

=== Michael_acceptor_1 해당 승인약물 ===
  TRETINOIN: CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1
  ETHACRYNIC ACID: C=C(CC)C(=O)c1ccc(OCC(=O)O)c(Cl)c1Cl
  ETRETINATE: CCOC(=O)/C=C(C)/C=C/C=C(C)/C=C/c1c(C)cc(OC)c(C)c1C
  OZAGREL: O=C(O)/C=C/c1ccc(Cn2ccnc2)cc1
  SUNITINIB: CCN(CC)CCNC(=O)c1c(C)[nH]c(/C=C2\C(=O)Nc3ccc(F)cc32)c1C
  ISOTRETINOIN: CC1=C(/C=C/C(C)=C/C=C/C(C)=C\C(=O)O)C(C)(C)CCC1
  ALITRETINOIN: CC1=C(/C=C/C(C)=C\C=C\C(C)=C\C(=O)O)C(C)(C)CCC1
  MUPIROCIN: C/C(=C\C(=O)OCCCCCCCCC(=O)O)C[C@@H]1OC[C@H](C[C@@H]2O[C@H]2[
  CILASTATIN: CC1(C)C[C@@H]1C(=O)N/C(=C\CCCCSC[C@H](N)C(=O)O)C(=O)O

=== nitro_group 해당 승인약물 ===
  ISOSORBIDE DINITRATE: O=[N+]([O-])O[C@H]1CO[C@H]2[C@@H]1OC[C@H]2O[N+](=O)[O-]
  CLONAZEPAM: O=C1CN=C(c2ccccc2Cl)c2cc([N+](=O)[O-])ccc2N1
  FLUNITRAZEPAM: CN1C(=O)CN=C(c2ccccc2F)c2cc([N+](=O)[O-])ccc21
  NITRAZEPAM: O=C1CN=C(c2ccccc2)c2cc([N+](=O)[O-])ccc2N1
  BENZNIDAZOLE: O=C(Cn1ccnc1[N+](=O)[O-])NCc1ccccc1
  NITROFURANTOIN: O=C1CN(/N=C/c2ccc([N+](=O)[O-])o2)

In [16]:
remaining_rules = ["alkyl_halide", "aniline", "Sulfonic_acid_2", "catechol", "Thiocarbonyl_group"]

for rule in remaining_rules:
    print(f"\n=== {rule} 해당 승인약물 ===")
    for d in drugs_with_names:
        problems = detect_toxicophores(d['smiles'])
        if any(p['rule_name'] == rule for p in problems):
            print(f"  {d['name']}: {d['smiles'][:60]}")


=== alkyl_halide 해당 승인약물 ===
  MECHLORETHAMINE: CN(CCCl)CCCl
  CYCLOPHOSPHAMIDE ANHYDROUS: O=P1(N(CCCl)CCCl)NCCCO1
  CARMUSTINE: O=NN(CCCl)C(=O)NCCCl
  LOMUSTINE: O=NN(CCCl)C(=O)NC1CCCCC1
  CHLORAMBUCIL: O=C(O)CCCc1ccc(N(CCCl)CCCl)cc1
  LINDANE: Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@@H](Cl)[C@H](Cl)[C@H]1Cl
  TRICHLOROETHANE: CC(Cl)(Cl)Cl
  CHLORAMPHENICOL: O=C(N[C@H](CO)[C@H](O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl
  PHENOXYBENZAMINE: CC(COc1ccccc1)N(CCCl)Cc1ccccc1

=== aniline 해당 승인약물 ===
  SULFANILAMIDE: Nc1ccc(S(N)(=O)=O)cc1
  SULFATHIAZOLE: Nc1ccc(S(=O)(=O)Nc2nccs2)cc1
  SULFAMERAZINE: Cc1ccnc(NS(=O)(=O)c2ccc(N)cc2)n1
  SULFADIAZINE: Nc1ccc(S(=O)(=O)Nc2ncccn2)cc1
  SULFAMETHOXAZOLE: Cc1cc(NS(=O)(=O)c2ccc(N)cc2)no1
  SULFAMETHAZINE: Cc1cc(C)nc(NS(=O)(=O)c2ccc(N)cc2)n1
  SULFISOXAZOLE: Cc1noc(NS(=O)(=O)c2ccc(N)cc2)c1C
  SULFAMETHOXYPYRIDAZINE: COc1ccc(NS(=O)(=O)c2ccc(N)cc2)nn1
  SULFACETAMIDE: CC(=O)NS(=O)(=O)c1ccc(N)cc1
  AMINOHIPPURIC ACID: Nc1ccc(C(=O)NCC(=O)O)cc1
  AMINOGLUTETHIMIDE: CCC1(c2c

In [17]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "[참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 "
                          "항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 "
                          "치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 "
                          "치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 "
                          "환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "[참고] 에타크린산처럼 시스테인 잔기와의 공유결합 자체가 "
                          "작용 메커니즘인 공유결합 억제제(covalent inhibitor) "
                          "계열에는 본 경고가 그대로 적용되지 않을 수 있음. || "
                          "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "[참고] 메클로르에타민, 사이클로포스파미드, 카머스틴, "
                          "클로람부실 등 알킬화 항암제는 DNA 알킬화(반응성) 자체가 "
                          "세포독성 치료 메커니즘이므로, 이 계열에는 본 치환이 "
                          "적절하지 않음. || 이탈기를 제거해 알킬화 반응성을 없앰, "
                          "극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NH2]c1ccc([#6,#7,#8,#16])cc1",
        "target_idx_in_pattern": 0,
        "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
        "anchor_indices_in_pattern": (0, 5),
        "candidates": [
            {"edit_type": "add_substituent", "param": "C(=O)C",
             "target_idx_in_pattern": 0,
             "name": "acetamide (acylated amine)",
             "rationale": "[참고] 설파계 항생제(설파닐아마이드, 설파메톡사졸 등)와 "
                          "프로카인아마이드처럼 아닐린 골격이 반응성 대사가 아닌 "
                          "안정적 형태로 널리 처방되어 온 사례가 다수 있음. 이 경우 "
                          "특이체질 반응은 드물고 예측이 어려워, 본 경고를 절대적 "
                          "배제 기준이 아닌 참고 신호로 해석해야 함. || 1차 방향족 "
                          "아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"edit_type": "replace_ring", "param": "[*:1]C12CC(C1)(C2)[*:2]",
             "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
             "anchor_indices_in_pattern": (0, 5),
             "name": "BCP (bicyclo[1.1.1]pentane)",
             "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic "
                          "탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive "
                          "metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 "
                          "생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 "
                          "(문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, "
                          "실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 "
                          "변화 폭이 크지만, 물성 개선 효과도 더 큼"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "[참고] 암페타민 설페이트, 사퀴나비르 메실레이트처럼 "
                          "일부 승인약물에서 설폰산/설폰산 유사기는 활성 골격이 "
                          "아니라 염(salt) 형성을 위한 카운터이온으로만 존재함. "
                          "이 경우 본 규칙이 다루는 '독성 유발 골격'과 무관하므로, "
                          "치환 대상 여부를 판단하기 전에 이 산이 활성 골격의 "
                          "일부인지 염 형성용인지 구분이 필요함. || 생리적 pH에서 "
                          "이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. "
                          "설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 "
                          "경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX3;!$(C(N)(N)=N)]=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요. "
                          "구아니딘(N-C(=N)-N, 공명구조로 일반 이민과 반응성이 다름)은 "
                          "이 SMARTS에서 명시적으로 제외함"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 도파민, 에피네프린, 이소프로테레놀 등 카테콜아민류 "
                          "약물은 카테콜 구조 자체가 아드레날린/도파민 수용체 결합에 "
                          "필수적인 약효 골격이므로, 이 경우 본 치환은 독성 감소가 "
                          "아니라 약효 상실로 이어짐. || 인체의 COMT(catechol-O-"
                          "methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 "
                          "해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 "
                          "경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 "
                          "예정: ScienceDirect catechol overview, PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "[참고] 티오펜탈·티아밀랄(치오바르비투레이트, C=S가 지용성 "
                          "증가로 빠른 마취효과에 기여)과 티오구아닌(퓨린 유사 항대사물, "
                          "황이 작용기전에 필수)처럼 황 원자가 약효/효력에 직접 "
                          "기여하는 경우가 있어, 이 계열에는 본 치환이 부적절할 수 "
                          "있음. || 황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 "
                          "흔한 bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 "
                          "작용기 특유의 대사/독성 우려를 낮춤 (검증 필요, "
                          "thiourea->urea 치환 논리와 동일 계열)"},
        ],
    },
    "thiol_2": {
        "problem_smarts": "[SX2H1]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 "
                          "제거하면서, 극성·수소결합 특성을 유사하게 유지함"},
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "티올을 아마이드로 대체하여 반응성을 낮추면서 약물유사 골격에서 "
                          "흔히 쓰이는 안정적 작용기로 전환 (검증 필요)"},
        ],
    },
    "thiol_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=S)[SX1-]",
        "candidates": [
            {"edit_type": "replace_multi",
             "param": [
                 {"idx_in_pattern": 1, "new_element": 8, "new_charge": 0},
                 {"idx_in_pattern": 2, "new_element": 7, "new_charge": 0},
             ],
             "name": "carbamate (O,N replacing S,S)",
             "rationale": "디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 "
                          "전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 "
                          "능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 "
                          "독성 기전)을 제거함 (검증 필요)"},
        ],
    },
    "het-C-het_not_in_ring": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4](O)(O)",
        "candidates": [
            {"edit_type": "remove_substituent",
             "center_idx_in_pattern": 0,
             "remove_idx_in_pattern": 1,
             "upgrade_bond_to_idx_in_pattern": 2,
             "name": "ketone/ester (one alkoxy removed, C=O formed)",
             "rationale": "아세탈/케탈 또는 오르토에스터(탄소 하나에 알콕시기 2개 "
                          "이상)는 가수분해에 민감하여 반응성 카르보닐(케톤/알데히드)로 "
                          "쉽게 분해되며 대사 불안정성을 일으킴. 알콕시기 하나를 제거하고 "
                          "남은 산소를 카르보닐로 승격시켜, 가수분해로 어차피 도달할 "
                          "안정한 최종 형태로 미리 전환함 (검증 필요)"},
        ],
    },
    "hydroquinone": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H]c1ccc([OX2H,NX3H1,NX3H2])cc1",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 아세트아미노펜은 정상 용량에서는 안전하며 과다복용 "
                          "시에만 위험한 용량 의존적 사례임. 본 시스템은 치료지수를 "
                          "고려하지 않으므로, 아트로핀·디곡신·와파린처럼 좁은 치료지수를 "
                          "가진 기존 약물 전반에 유사하게 적용되는 한계임. || 파라 "
                          "위치에 OH와 (OH 또는 NH)가 있는 구조(하이드로퀴논/파라-"
                          "아미노페놀 계열)는 산화되어 파라-퀴논 또는 파라-퀴논이민(예: "
                          "아세트아미노펜의 NAPQI)을 형성, 글루타치온 고갈과 단백질 "
                          "공유결합을 통한 간독성 위험이 있음"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [18]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

print(propose_fix("Oc1ccc(O)cc1", "hydroquinone", candidate_idx=0))
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix("N#CC(C#N)=Cc1ccc(O)c(O)c1", "catechol", candidate_idx=0))

{'new_smiles': 'COc1ccc(O)cc1', 'candidate_used': 'methoxy', 'rationale': '[참고] 아세트아미노펜은 정상 용량에서는 안전하며 과다복용 시에만 위험한 용량 의존적 사례임. 본 시스템은 치료지수를 고려하지 않으므로, 아트로핀·디곡신·와파린처럼 좁은 치료지수를 가진 기존 약물 전반에 유사하게 적용되는 한계임. || 파라 위치에 OH와 (OH 또는 NH)가 있는 구조(하이드로퀴논/파라-아미노페놀 계열)는 산화되어 파라-퀴논 또는 파라-퀴논이민(예: 아세트아미노펜의 NAPQI)을 형성, 글루타치온 고갈과 단백질 공유결합을 통한 간독성 위험이 있음', 'is_valid': True}
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '[참고] 메클로르에타민, 사이클로포스파미드, 카머스틴, 클로람부실 등 알킬화 항암제는 DNA 알킬화(반응성) 자체가 세포독성 치료 메커니즘이므로, 이 계열에는 본 치환이 적절하지 않음. || 이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
{'new_smiles': 'COc1ccc(C=C(C#N)C#N)cc1O', 'candidate_used': 'methoxy', 'rationale': '[참고] 도파민, 에피네프린, 이소프로테레놀 등 카테콜아민류 약물은 카테콜 구조 자체가 아드레날린/도파민 수용체 결합에 필수적인 약효 골격이므로, 이 경우 본 치환은 독성 감소가 아니라 약효 상실로 이어짐. || 인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 예정: ScienceDirect catechol overview, PMC6643002 등 참고)', 'is_valid': 

In [19]:
import inspect
print(inspect.getsource(ask_llm_which_candidate_to_use))

def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    if len(candidates) == 1:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 여러 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

이 중 이 분자 맥락에서 가장 적절한 후보를 선택하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수), "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_respons

In [21]:
!pip install openai -q
from openai import OpenAI
dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(api_key=dashscope_key, base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1")
print("Qwen 클라이언트 준비 완료")

Qwen 클라이언트 준비 완료


In [22]:
# 1) 함수 호출 (평소 방식)
decision_check = ask_llm_which_candidate_to_use(client_qwen, "qwen3.8-max-preview",
                                                  "Nc1ccc(S(N)(=O)=O)cc1", "aniline",
                                                  client_type="openai_compatible")
print("=== 함수 결과 ===")
print(decision_check)

# 2) 직접 호출 - 실제 프롬프트와 원문 응답을 그대로 확인
info = get_replacement_candidates("aniline")
candidate_info = [{"idx": i, "name": c['name'], "rationale": c['rationale']} for i, c in enumerate(info['candidates'])]

import json
prompt_direct = f"""당신은 신약개발 화학자입니다. 다음 분자에서 'aniline' 문제를
해결하기 위한 여러 치환 후보가 있습니다.

분자 SMILES: Nc1ccc(S(N)(=O)=O)cc1

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

이 중 이 분자 맥락에서 가장 적절한 후보를 선택하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수), "reason": "선택 이유 한 문장"}}
"""

print("\n=== 실제 프롬프트 ===")
print(prompt_direct)

response_direct = client_qwen.chat.completions.create(
    model="qwen3.8-max-preview",
    messages=[{"role": "user", "content": prompt_direct}]
)
print("\n=== LLM 원문 응답 ===")
print(response_direct.choices[0].message.content)

=== 함수 결과 ===
{'candidate_idx': 0, 'reason': '설파닐아마이드의 핵심 골격을 유지하면서 1차 방향족 아민의 N-hydroxylation 경로를 직접 차단하는 최소·표적 변형이기 때문이다.'}

=== 실제 프롬프트 ===
당신은 신약개발 화학자입니다. 다음 분자에서 'aniline' 문제를
해결하기 위한 여러 치환 후보가 있습니다.

분자 SMILES: Nc1ccc(S(N)(=O)=O)cc1

치환 후보들:
[
  {
    "idx": 0,
    "name": "acetamide (acylated amine)",
    "rationale": "[참고] 설파계 항생제(설파닐아마이드, 설파메톡사졸 등)와 프로카인아마이드처럼 아닐린 골격이 반응성 대사가 아닌 안정적 형태로 널리 처방되어 온 사례가 다수 있음. 이 경우 특이체질 반응은 드물고 예측이 어려워, 본 경고를 절대적 배제 기준이 아닌 참고 신호로 해석해야 함. || 1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"
  },
  {
    "idx": 1,
    "name": "BCP (bicyclo[1.1.1]pentane)",
    "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic 탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 (문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, 실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 변화 폭이 크지만, 물성 개선 효과도 더 큼"
  }
]

이 중 이 분자 맥락에서 가장 적절한 후보를 선택하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{"candidate_idx": 선택한 후보의 idx(정

In [23]:
def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    if len(candidates) == 1:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 여러 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

각 후보의 rationale에 "[참고]"로 시작하는 문구가 있다면, 이는 "이 골격이
실제 승인 약물에서 반응성이 아닌 안정적 형태로 널리 쓰인 사례가 있으니,
경고를 절대적 기준이 아닌 참고 신호로 해석하라"는 뜻입니다. 이 경우 먼저
"이 분자가 그 참고사항이 가리키는 안전한 사용 사례와 실제로 유사한지"를
판단하세요. 만약 유사하다고 판단되면, 변화 폭이 더 작은 후보를 우선
고려하거나, 치환 자체가 불필요할 수 있음을 reason에 명시하세요.

이 중 이 분자 맥락에서 가장 적절한 후보를 선택하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수), "reason": "선택 이유 한 문장(참고사항을 고려했다면 그 판단 근거 포함)"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_response(text, fallback)

    if not isinstance(result.get('candidate_idx'), int) or not (0 <= result['candidate_idx'] < len(candidates)):
        return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
    return result

In [24]:
!cat src/tools/agent.py

import json
from src.tools.replacement_library import get_replacement_candidates


def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback


def

In [25]:
%%writefile src/tools/agent.py
import json
from src.tools.replacement_library import get_replacement_candidates


def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback


def ask_llm_which_problem_to_fix(client, model_name, smiles, problems, client_type="gemini"):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    return _parse_json_response(text, fallback)


def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청.
    candidate의 rationale에 '[참고]'로 시작하는 문구가 있으면, 이는 실제 승인약물
    사례에서 이 골격이 안전하게 쓰인 경우가 있다는 뜻이므로, LLM이 이를 단순
    배경정보가 아니라 치환 필요성 자체에 대한 판단 근거로 명시적으로 고려하도록
    프롬프트에서 지시한다."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    if len(candidates) == 1:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 여러 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

각 후보의 rationale에 "[참고]"로 시작하는 문구가 있다면, 이는 "이 골격이
실제 승인 약물에서 반응성이 아닌 안정적 형태로 널리 쓰인 사례가 있으니,
경고를 절대적 기준이 아닌 참고 신호로 해석하라"는 뜻입니다. 이 경우 먼저
"이 분자가 그 참고사항이 가리키는 안전한 사용 사례와 실제로 유사한지"를
판단하세요. 만약 유사하다고 판단되면, 변화 폭이 더 작은 후보를 우선
고려하거나, 치환 자체가 불필요할 수 있음을 reason에 명시하세요.

이 중 이 분자 맥락에서 가장 적절한 후보를 선택하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수), "reason": "선택 이유 한 문장(참고사항을 고려했다면 그 판단 근거 포함)"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_response(text, fallback)

    if not isinstance(result.get('candidate_idx'), int) or not (0 <= result['candidate_idx'] < len(candidates)):
        return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
    return result

Overwriting src/tools/agent.py


In [26]:
importlib.reload(src.tools.agent)
from src.tools.agent import ask_llm_which_candidate_to_use

# 참고사항이 있는 규칙들 + 실제로 그 참고사항에 해당할 만한 분자들
test_cases_reference = [
    ("aniline", "Nc1ccc(S(N)(=O)=O)cc1"),           # 설파닐아마이드 유사
    ("aniline", "Cc1cc(-c2cc(C)c(N)c(C)c2)cc(C)c1N"),  # ortho 치환 많은 아닐린 (BCP 부적합 사례)
    ("catechol", "NCCc1ccc(O)c(O)c1"),                # 도파민과 매우 유사
    ("catechol", "N#CC(C#N)=Cc1ccc(O)c(O)c1"),        # 카테콜아민과 무관한 일반 분자
    ("Michael_acceptor_1", "C=C(CC)C(=O)c1ccc(OCC(=O)O)c(Cl)c1Cl"),  # 에타크린산과 유사
    ("Michael_acceptor_1", "C=CC(=O)OCCCCCC"),        # 단순 아크릴레이트, 무관
]

print("=== 개선된 프롬프트로 재테스트 ===\n")
for rule, smi in test_cases_reference:
    decision = ask_llm_which_candidate_to_use(client_qwen, "qwen3.8-max-preview", smi, rule, client_type="openai_compatible")
    candidate_name = get_replacement_candidates(rule)['candidates'][decision['candidate_idx']]['name']
    print(f"[{rule}] {smi[:50]}")
    print(f"  선택: {candidate_name} (idx={decision['candidate_idx']})")
    print(f"  이유: {decision['reason']}\n")

=== 개선된 프롬프트로 재테스트 ===

[aniline] Nc1ccc(S(N)(=O)=O)cc1
  선택: acetamide (acylated amine) (idx=0)
  이유: 이 분자는 설파닐아마이드로서 [참고]의 설파계 항생제 안정적 사용 사례와 직접 유사해 aniline 경고를 절대 배제 기준으로 보지 않으며, 치환이 필요하다면 변화 폭이 더 작은 acetamide 아실화를 선택한다.

[aniline] Cc1cc(-c2cc(C)c(N)c(C)c2)cc(C)c1N
  선택: acetamide (acylated amine) (idx=0)
  이유: 이 분자는 참고사항이 가리키는 안정적 아닐린 함유 약물과 유사하게 치환된 방향족 아민으로 판단되므로, 골격을 크게 바꾸는 BCP보다 N-hydroxylation을 직접 차단하는 소폭의 아미드화가 더 적절하다.

[catechol] NCCc1ccc(O)c(O)c1
  선택: methoxy (idx=0)
  이유: 유일한 후보

[catechol] N#CC(C#N)=Cc1ccc(O)c(O)c1
  선택: methoxy (idx=0)
  이유: 유일한 후보

[Michael_acceptor_1] C=C(CC)C(=O)c1ccc(OCC(=O)O)c(Cl)c1Cl
  선택: saturated (C-C single bond) (idx=0)
  이유: 유일한 후보

[Michael_acceptor_1] C=CC(=O)OCCCCCC
  선택: saturated (C-C single bond) (idx=0)
  이유: 유일한 후보



In [27]:
%%writefile src/tools/agent.py
import json
from src.tools.replacement_library import get_replacement_candidates


def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback


def ask_llm_which_problem_to_fix(client, model_name, smiles, problems, client_type="gemini"):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    return _parse_json_response(text, fallback)


def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청.

    candidate의 rationale 중 하나라도 '[참고]'로 시작하는 문구가 있으면,
    이는 실제 승인약물 사례에서 이 골격이 안전하게 쓰인 경우가 있다는 뜻이므로,
    candidate가 1개뿐이더라도(원래는 LLM 호출을 건너뛰던 경우) 반드시 LLM에게
    판단을 맡긴다. 이 경우 LLM은 candidate_idx로 -1을 반환하여 "치환을
    보류하고 사람(연구자) 검토가 필요하다"고 명시적으로 표시할 수 있다.
    """
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    has_caution = any('[참고]' in c.get('rationale', '') for c in candidates)

    if len(candidates) == 1 and not has_caution:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

각 후보의 rationale에 "[참고]"로 시작하는 문구가 있다면, 이는 "이 골격이
실제 승인 약물에서 반응성이 아닌 안정적 형태로 널리 쓰인 사례가 있으니,
경고를 절대적 기준이 아닌 참고 신호로 해석하라"는 뜻입니다. 이 경우 먼저
"이 분자가 그 참고사항이 가리키는 안전한 사용 사례와 실제로 유사한지"를
판단하세요.
- 유사하다고 판단되면서, 후보가 여러 개라면 변화 폭이 더 작은 후보를 선택하세요.
- 유사하다고 판단되고, 치환 자체가 불필요하다고 볼 만큼 뚜렷하다면,
  candidate_idx를 -1로 답해 "치환 보류, 사람 검토 필요"를 표시하세요.
- 참고사항이 없거나 이 분자가 그 사례와 유사하지 않다면, 평소대로 가장
  적절한 후보를 선택하세요.

반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수, 또는 보류 시 -1), "reason": "판단 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_response(text, fallback)

    idx = result.get('candidate_idx')
    if not isinstance(idx, int) or not (-1 <= idx < len(candidates)):
        return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
    return result

Overwriting src/tools/agent.py


In [28]:
importlib.reload(src.tools.agent)
importlib.reload(src.tools.molecule_editor)
from src.tools.agent import ask_llm_which_candidate_to_use
from src.tools.molecule_editor import iterative_fix_loop

# 도파민 유사 분자로 전체 루프 테스트 (catechol이 이제 candidate 1개라도 LLM 호출됨)
test_dopamine_like = "NCCc1ccc(O)c(O)c1"
result_dopamine = iterative_fix_loop(test_dopamine_like, max_iterations=5,
                                      llm_client=client_qwen, llm_model="qwen3.8-max-preview",
                                      llm_client_type="openai_compatible")
print("상태:", result_dopamine['status'])
for h in result_dopamine['history']:
    print(h)
print("\nskipped_details:")
for d in result_dopamine.get('skipped_details', []):
    print(f"  [{d['rule_name']}] {d['reason']}")

상태: success
{'step': 0, 'smiles': 'NCCc1ccc(O)c(O)c1', 'problems': [{'rule_name': 'catechol_A(92)', 'atom_indices': [3, 4, 5, 6, 7, 8, 9, 10]}, {'rule_name': 'catechol', 'atom_indices': [3, 4, 5, 6, 7, 8, 9, 10]}]}
{'step': 1, 'smiles': 'COc1ccc(CCN)cc1O', 'fixed_rule': 'catechol', 'problem_reason': '유일한 치환 가능 후보', 'candidate_used': 'methoxy', 'candidate_reason': '이 분자는 도파민과 같은 카테콜아민 계열로 카테콜이 약효에 필수적이므로 메톡시 치환은 독성 감소보다 약효 상실을 초래할 수 있어 사람 검토가 필요하다.', 'problems': []}

skipped_details:
  [catechol_A(92)] 라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 'catechol_A(92)'로 진단했으며, 매치된 원자 인덱스는 [3, 4, 5, 6, 7, 8, 9, 10](원소: ['C', 'C', 'C', 'C', 'O', 'C', 'O', 'C'])입니다. 이 구조에 대한 치환 규칙을 replacement_library.py에 추가하면 자동으로 처리 가능합니다.


In [30]:
!git add src/tools/agent.py src/tools/molecule_editor.py
!git add src/tools/replacement_library.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/agent.py
	modified:   src/tools/replacement_library.py



In [31]:
!git commit -m "Add candidate_idx=-1 (defer to human review) mechanism: rationale with '[참고]' now forces LLM consultation even for single-candidate rules, and LLM can flag the substitution as inappropriate (e.g. catechol on dopamine-like molecules where the toxicophore is essential for activity). Add hydroquinone rule and [참고] caution notes across 8 rules based on ChEMBL cross-reference of approved drugs with therapeutic-index/mechanism-is-toxicity cases."
!git push origin main

[main 44db34e] Add candidate_idx=-1 (defer to human review) mechanism: rationale with '[참고]' now forces LLM consultation even for single-candidate rules, and LLM can flag the substitution as inappropriate (e.g. catechol on dopamine-like molecules where the toxicophore is essential for activity). Add hydroquinone rule and [참고] caution notes across 8 rules based on ChEMBL cross-reference of approved drugs with therapeutic-index/mechanism-is-toxicity cases.
 2 files changed, 87 insertions(+), 22 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 3.44 KiB | 880.00 KiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   7f16df7..44db34e  main -> main


In [32]:
test_cases_wide = [
    ("aniline", "Nc1ccc(S(N)(=O)=O)cc1", "설파닐아마이드 유사"),
    ("aniline", "CCN(CC)CCNC(=O)c1ccc(N)cc1", "프로카인아마이드 유사"),
    ("aniline", "Cc1ccc(N)cc1", "단순 톨루이딘 (참고사항 무관해야 함)"),
    ("catechol", "NCCc1ccc(O)c(O)c1", "도파민"),
    ("catechol", "CNC[C@H](O)c1ccc(O)c(O)c1", "에피네프린 유사"),
    ("catechol", "N#CC(C#N)=Cc1ccc(O)c(O)c1", "카테콜아민과 무관"),
    ("Michael_acceptor_1", "C=C(CC)C(=O)c1ccc(OCC(=O)O)c(Cl)c1Cl", "에타크린산"),
    ("Michael_acceptor_1", "C=CC(=O)OCCCCCC", "단순 아크릴레이트"),
    ("nitro_group", "Cc1ncc([N+](=O)[O-])n1CCO", "메트로니다졸"),
    ("nitro_group", "O=C1CN=C(c2ccccc2)c2cc([N+](=O)[O-])ccc2N1", "니트라제팜(벤조디아제핀)"),
    ("Thiocarbonyl_group", "CCCC(C)C1(CC)C(=O)NC(=S)NC1=O", "티오펜탈"),
]

print("=== 참고사항 인지 테스트 (11개 분자) ===\n")
for rule, smi, label in test_cases_wide:
    decision = ask_llm_which_candidate_to_use(client_qwen, "qwen3.8-max-preview", smi, rule, client_type="openai_compatible")
    idx = decision['candidate_idx']
    action = "🛑 보류(사람검토)" if idx == -1 else f"진행(idx={idx})"
    print(f"[{rule}] {label}")
    print(f"  판단: {action}")
    print(f"  이유: {decision['reason']}\n")

=== 참고사항 인지 테스트 (11개 분자) ===

[aniline] 설파닐아마이드 유사
  판단: 🛑 보류(사람검토)
  이유: 이 분자는 참고사항이 언급한 설파닐아마이드(승인 설파계 항생제) 그 자체로 안정적 사용 사례와 직접 일치하므로 자동 치환보다 사람 검토가 적절하다.

[aniline] 프로카인아마이드 유사
  판단: 🛑 보류(사람검토)
  이유: 이 분자는 참고 사례로 언급된 프로카인아마이드와 동일한 승인 약물 골격이므로 자동 치환보다 사람 검토가 적절합니다.

[aniline] 단순 톨루이딘 (참고사항 무관해야 함)
  판단: 진행(idx=0)
  이유: p-톨루이딘은 설파닐아마이드/프로카인아마이드 같은 안정적 약물 사례와 직접 유사하지 않아, 평소대로 N-하이드록실화를 차단하는 최소 변화 아실화를 선택.

[catechol] 도파민
  판단: 🛑 보류(사람검토)
  이유: 이 분자는 도파민/카테콜아민 계열의 핵심 약효 골격으로 카테콜이 수용체 결합에 필수적이므로 메톡시 치환은 약효 상실 가능성이 커 사람 검토가 필요하다.

[catechol] 에피네프린 유사
  판단: 🛑 보류(사람검토)
  이유: 이 분자는 에피네프린과 같은 카테콜아민 골격으로 카테콜이 수용체 결합에 필수적이므로 메톡시 치환은 약효 상실을 초래해 사람 검토가 필요하다.

[catechol] 카테콜아민과 무관
  판단: 진행(idx=0)
  이유: 이 분자는 카테콜아민 계열의 필수 약효 골격과 유사하지 않으므로, 카테콜 산화 독성을 줄이기 위한 메톡시 치환이 적절합니다.

[Michael_acceptor_1] 에타크린산
  판단: 🛑 보류(사람검토)
  이유: 이 분자는 승인 약물 에타크린산과 동일한 α,β-불포화 카르보닐 기반 공유결합성 골격이므로 Michael acceptor 경고를 절대 결격으로 보지 않고 치환 보류 후 사람 검토가 필요하다.

[Michael_acceptor_1] 단순 아크릴레이트
  판단: 진행(idx=0)
  이유: 이 분자는 단순 알킬 아크릴레이트

In [33]:
# catechol_A(92)가 catechol과 다른지 확인
examples_ca = None
for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] == 'catechol_A(92)':
            examples_ca = (s, p['atom_indices'])
            break
    if examples_ca:
        break

print("catechol_A(92) 예시:", examples_ca[0] if examples_ca else "못찾음(valid set 내)")
if examples_ca:
    mol = Chem.MolFromSmiles(examples_ca[0])
    for idx in examples_ca[1]:
        atom = mol.GetAtomWithIdx(idx)
        print(f"  idx={idx}: {atom.GetSymbol()} (이웃: {[n.GetSymbol() for n in atom.GetNeighbors()]})")

# 도파민으로도 비교
print("\n도파민에서 catechol_A(92) 여부:")
print([p for p in detect_toxicophores("NCCc1ccc(O)c(O)c1") if 'catechol' in p['rule_name']])

catechol_A(92) 예시: O=C(C=Cc1ccc(O)c(O)c1)O[C@@H]1C[C@@](OC(=O)C=Cc2ccc(O)c(O)c2)(C(=O)O)C[C@@H](O)[C@@H]1O
  idx=4: C (이웃: ['C', 'C', 'C'])
  idx=5: C (이웃: ['C', 'C'])
  idx=6: C (이웃: ['C', 'C'])
  idx=7: C (이웃: ['C', 'O', 'C'])
  idx=8: O (이웃: ['C'])
  idx=9: C (이웃: ['C', 'O', 'C'])
  idx=10: O (이웃: ['C'])
  idx=11: C (이웃: ['C', 'C'])

도파민에서 catechol_A(92) 여부:
[{'rule_name': 'catechol_A(92)', 'atom_indices': [3, 4, 5, 6, 7, 8, 9, 10]}, {'rule_name': 'catechol', 'atom_indices': [3, 4, 5, 6, 7, 8, 9, 10]}]


In [34]:
# 여러 카테콜 함유 분자에서 두 규칙이 항상 같은 원자를 가리키는지 확인
test_catechol_check = ["NCCc1ccc(O)c(O)c1", "N#CC(C#N)=Cc1ccc(O)c(O)c1", "CNC[C@H](O)c1ccc(O)c(O)c1"]
for s in test_catechol_check:
    problems = detect_toxicophores(s)
    catechol_matches = [p for p in problems if 'catechol' in p['rule_name']]
    print(f"{s[:40]}: {catechol_matches}")

NCCc1ccc(O)c(O)c1: [{'rule_name': 'catechol_A(92)', 'atom_indices': [3, 4, 5, 6, 7, 8, 9, 10]}, {'rule_name': 'catechol', 'atom_indices': [3, 4, 5, 6, 7, 8, 9, 10]}]
N#CC(C#N)=Cc1ccc(O)c(O)c1: [{'rule_name': 'catechol_A(92)', 'atom_indices': [6, 7, 8, 9, 10, 11, 12, 13]}, {'rule_name': 'catechol', 'atom_indices': [6, 7, 8, 9, 10, 11, 12, 13]}]
CNC[C@H](O)c1ccc(O)c(O)c1: [{'rule_name': 'catechol_A(92)', 'atom_indices': [5, 6, 7, 8, 9, 10, 11, 12]}, {'rule_name': 'catechol', 'atom_indices': [5, 6, 7, 8, 9, 10, 11, 12]}]


In [35]:
%%writefile src/tools/toxicophore_detector.py
from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")
_guanidine_pattern = Chem.MolFromSmarts("[$(C(N)(N)=N)]")

# PAINS와 BRENK 양쪽에 동일 화학구조를 잡는 중복 규칙명이 있는 경우,
# 우리 라이브러리 기준 이름으로 통일 (동일 원자 인덱스로 확인된 것만)
_DUPLICATE_RULE_MAP = {
    "catechol_A(92)": "catechol",
}


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH), 구아니딘(N-C(=N)-N), 일반 이민(C=N-R)을
    모두 포함하는 넓은 카테고리이므로, 실제 매치 부분의 화학적 맥락을
    확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_oxime"
    if mol.HasSubstructMatch(_guanidine_pattern):
        matches = mol.GetSubstructMatches(_guanidine_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_guanidine"
    return "imine_1_general"


def detect_toxicophores(smiles: str) -> list[dict]:
    """
    분자의 SMILES를 받아, FilterCatalog(PAINS+BRENK)에 매치되는
    문제 구조(toxicophore)들을 찾아서 규칙 이름과 해당 원자 인덱스를 반환.
    imine_1은 옥심/구아니딘/일반이민 하위형으로 세분화하여 반환한다.
    aniline은 FilterCatalog의 단순 [NH2] 탐지 대신, replacement_library의
    확장된 패턴(para-치환 벤젠 포함)을 그대로 사용해 재정의한다.
    PAINS/BRENK가 동일 원자를 서로 다른 이름으로 중복 보고하는 경우
    (예: catechol_A(92) == catechol), 라이브러리 기준 이름으로 통일하고
    중복 항목은 제거한다.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    results = []
    seen_entries = set()

    for entry in _catalog.GetMatches(mol):
        for fm in entry.GetFilterMatches(mol):
            atom_indices = sorted(set(mol_idx for _, mol_idx in fm.atomPairs))
            rule_name = entry.GetDescription()

            if rule_name == "imine_1":
                rule_name = _refine_imine1(mol, atom_indices)
            elif rule_name == "aniline":
                continue
            elif rule_name in _DUPLICATE_RULE_MAP:
                rule_name = _DUPLICATE_RULE_MAP[rule_name]

            dedup_key = (rule_name, tuple(atom_indices))
            if dedup_key in seen_entries:
                continue
            seen_entries.add(dedup_key)

            results.append({
                "rule_name": rule_name,
                "atom_indices": atom_indices,
            })

    from src.tools.replacement_library import get_replacement_candidates
    aniline_info = get_replacement_candidates("aniline")
    if aniline_info:
        aniline_pattern = Chem.MolFromSmarts(aniline_info["problem_smarts"])
        if mol.HasSubstructMatch(aniline_pattern):
            matches = mol.GetSubstructMatches(aniline_pattern)
            for match in matches:
                atom_indices = sorted(set(match))
                results.append({
                    "rule_name": "aniline",
                    "atom_indices": atom_indices,
                })

    return results

Overwriting src/tools/toxicophore_detector.py


In [36]:
importlib.reload(src.tools.toxicophore_detector)
from src.tools.toxicophore_detector import detect_toxicophores

for s in test_catechol_check:
    problems = detect_toxicophores(s)
    catechol_matches = [p for p in problems if 'catechol' in p['rule_name']]
    print(f"{s[:40]}: {catechol_matches}")

NCCc1ccc(O)c(O)c1: [{'rule_name': 'catechol', 'atom_indices': [3, 4, 5, 6, 7, 8, 9, 10]}]
N#CC(C#N)=Cc1ccc(O)c(O)c1: [{'rule_name': 'catechol', 'atom_indices': [6, 7, 8, 9, 10, 11, 12, 13]}]
CNC[C@H](O)c1ccc(O)c(O)c1: [{'rule_name': 'catechol', 'atom_indices': [5, 6, 7, 8, 9, 10, 11, 12]}]


In [37]:
!git add src/tools/toxicophore_detector.py
!git commit -m "Merge duplicate FilterCatalog entries: catechol_A(92) and catechol always matched identical atom indices across test cases (PAINS/BRENK reporting same structure with different names), unified to 'catechol' with dedup logic to prevent double-reporting in skipped_details/problems list"
!git push origin main

[main c4712d5] Merge duplicate FilterCatalog entries: catechol_A(92) and catechol always matched identical atom indices across test cases (PAINS/BRENK reporting same structure with different names), unified to 'catechol' with dedup logic to prevent double-reporting in skipped_details/problems list
 1 file changed, 18 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.05 KiB | 1.05 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   44db34e..c4712d5  main -> main


In [38]:
all_catalog_names_full = set()
for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    for x in p:
        all_catalog_names_full.add(x['rule_name'])

print(f"Valid set에서 실제로 발견되는 고유 규칙 수: {len(all_catalog_names_full)}개")
print(f"우리가 다루는 규칙 수: {len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])}개")
print(f"커버 비율(규칙 종류 기준): {len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])/len(all_catalog_names_full)*100:.1f}%")

Valid set에서 실제로 발견되는 고유 규칙 수: 99개
우리가 다루는 규칙 수: 15개
커버 비율(규칙 종류 기준): 15.2%


In [39]:
from collections import Counter

rule_freq_all = Counter()
for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    for x in p:
        rule_freq_all[x['rule_name']] += 1

covered = set(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
uncovered_freq = {r: c for r, c in rule_freq_all.items() if r not in covered}
sorted_uncovered = sorted(uncovered_freq.items(), key=lambda x: -x[1])

print("미커버 규칙 빈도 (상위 20개):")
cumulative = 0
for rule, count in sorted_uncovered[:20]:
    cumulative += count
    print(f"  {rule}: {count}개 (누적 {cumulative}개)")

print(f"\n미커버 규칙 전체(79개) 합계: {sum(uncovered_freq.values())}개")
print(f"상위 20개가 차지하는 비중: {cumulative}/{sum(uncovered_freq.values())*100:.0f}%" if False else "")
print(f"상위 20개 비중: {cumulative/sum(uncovered_freq.values())*100:.1f}%")

미커버 규칙 빈도 (상위 20개):
  Aliphatic_long_chain: 170개 (누적 170개)
  Oxygen-nitrogen_single_bond: 79개 (누적 249개)
  isolated_alkene: 65개 (누적 314개)
  phosphor: 33개 (누적 347개)
  quaternary_nitrogen_1: 28개 (누적 375개)
  quaternary_nitrogen_2: 23개 (누적 398개)
  triple_bond: 17개 (누적 415개)
  beta-keto/anhydride: 16개 (누적 431개)
  heavy_metal: 16개 (누적 447개)
  iodine: 14개 (누적 461개)
  azo_A(324): 13개 (누적 474개)
  diazo_group: 13개 (누적 487개)
  halogenated_ring_1: 10개 (누적 497개)
  Three-membered_heterocycle: 10개 (누적 507개)
  phenol_ester: 9개 (누적 516개)
  diketo_group: 8개 (누적 524개)
  imine_2: 7개 (누적 531개)
  quinone_A(370): 7개 (누적 538개)
  polyene: 6개 (누적 544개)
  stilbene: 6개 (누적 550개)

미커버 규칙 전체(79개) 합계: 720개

상위 20개 비중: 76.4%
